# Paper Explorer: Full Project Walkthrough

This notebook demonstrates every major feature of the package end to end: ingesting papers from arXiv into a local SQLite-backed library, semantic / keyword / hybrid search, cross-encoder reranking, metadata filtering, index rebuilding, and the two comparison visualizations.

Run `uv pip install -e .` from the project root before running this notebook.

Each section below is a runnable cell.

**Note:** the first cell that embeds text downloads a small sentence-transformers
model (~90MB); the first cell that reranks downloads a small cross-encoder
model (~80MB). Both are one-time downloads, cached locally afterward.

## 1. Imports and setup

In [ ]:
from paper_explorer import (
    ArxivClient,
    BM25Index,
    EmbeddingModel,
    HybridSearcher,
    IngestionService,
    PaperRepository,
    PaperSearcher,
    VectorIndex,
)
from paper_explorer.config import DEFAULT_DB_PATH, DEFAULT_ID_MAP_PATH, DEFAULT_INDEX_PATH
from paper_explorer.search.display import with_display_scores
from paper_explorer.search.pipeline import run_search
from paper_explorer.search.reranker import CrossEncoderReranker
from paper_explorer.viz.plots import plot_mode_overlap, plot_reranking_impact

QUERY_TOPIC = "transformer attention mechanism"
SEARCH_QUERY = "attention is all you need NLP"
CATEGORY = "cs.CL"

## 2. (Optional) Reset the local library

Uncomment to start completely fresh. Skip this cell to keep any existing data.

In [20]:
# repository = PaperRepository(DEFAULT_DB_PATH)
# repository.reset()
# print("Reset complete")

## 3. Ingest papers from arXiv

Fetches up to 200 papers matching the topic (restricted to the `cs.CL` category for a focused library), computes an embedding for each paper's title + abstract, and stores everything in a local SQLite database.

Re-running this cell on the same topic is cheap: unchanged papers are skipped, not re-embedded (incremental, resumable ingestion).

In [8]:
repository = PaperRepository(DEFAULT_DB_PATH)
service = IngestionService(repository, client=ArxivClient())

summary = service.ingest(QUERY_TOPIC, max_results=200, category=CATEGORY)
print(summary)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7471.31it/s]
/Users/sujan/Desktop/paper_finder/paper_explorer/src/paper_explorer/embeddings/embedding_model.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self._model.get_sentence_embedding_dimension()


{'fetched': 200, 'new': 200, 'changed': 0, 'unchanged': 0, 'embedded': 200, 'embedding_failed': 0}


## 4. Rebuild and save the FAISS index

Cheap: reads already-computed vectors back out of SQLite. No model
inference involved.

In [9]:
index = service.rebuild_index()
index.save(DEFAULT_INDEX_PATH, DEFAULT_ID_MAP_PATH)
print(f"Indexed {len(index)} vectors")

Indexed 200 vectors


## 5. Inspect the library

In [10]:
stats = repository.stats()
for key, value in stats.items():
    print(f"{key}: {value}")

total_papers: 200
embedded: 200
pending: 0
stale: 0
failed: 0
embedding_model: all-MiniLM-L6-v2
embedding_dimension: 384
categories: ['cond-mat.mes-hall', 'cond-mat.mtrl-sci', 'cs.AI', 'cs.CC', 'cs.CL', 'cs.CV', 'cs.FL', 'cs.HC', 'cs.IR', 'cs.IT', 'cs.LG', 'cs.MM', 'cs.NE', 'cs.PF', 'cs.SD', 'cs.SI', 'eess.AS', 'eess.IV', 'eess.SP', 'math.AT', 'math.OC', 'physics.data-an', 'q-bio.NC', 'quant-ph', 'stat.ML']
date_range: ('2017-01-03', '2026-09-08')


## 6. Build the search components

BM25 is rebuilt fresh from whatever is currently in the repository  (cheap: pure tokenization, no ML). The FAISS index is loaded from disk. The embedding model loads once and is reused for every query below.

In [11]:
embedding_model = EmbeddingModel()
vector_index = VectorIndex.load(DEFAULT_INDEX_PATH, DEFAULT_ID_MAP_PATH)
searcher = PaperSearcher(store=repository, index=vector_index, embedding_model=embedding_model)

bm25_index = BM25Index()
bm25_index.build(repository.all())

hybrid_searcher = HybridSearcher(store=repository, bm25_index=bm25_index, searcher=searcher)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6894.23it/s]
/Users/sujan/Desktop/paper_finder/paper_explorer/src/paper_explorer/embeddings/embedding_model.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self._model.get_sentence_embedding_dimension()


## 7. Semantic search

In [12]:
semantic_results = run_search(hybrid_searcher, SEARCH_QUERY, mode="semantic", top_k=10)
for r in semantic_results:
    print(f"{r.rank:>2}. score={r.score:.3f}  {r.paper.title}")

 1. score=0.573  Area Attention
 2. score=0.523  Top-Theta Attention: Sparsifying Transformers by Compensated Thresholding
 3. score=0.518  Selective Attention Improves Transformer
 4. score=0.514  Attention Flows: Analyzing and Comparing Attention Mechanisms in Language Models
 5. score=0.514  Contrastive Attention Mechanism for Abstractive Sentence Summarization
 6. score=0.512  Skip-Layer Attention: Bridging Abstract and Detailed Dependencies in Transformers
 7. score=0.509  Do New Attention Mechanisms Actually Fix Attention Sinks at Million-Token Context?
 8. score=0.508  Analyzing the Structure of Attention in a Transformer Language Model
 9. score=0.507  AttentionViz: A Global View of Transformer Attention
10. score=0.506  H-Transformer-1D: Fast One-Dimensional Hierarchical Attention for Sequences


## 8. Keyword (BM25) search

In [13]:
keyword_results = run_search(hybrid_searcher, SEARCH_QUERY, mode="keyword", top_k=10)
for r in keyword_results:
    print(f"{r.rank:>2}. raw_bm25={r.score:.3f}  {r.paper.title}")

 1. raw_bm25=12.791  Attention Is Not All You Need: The Importance of Feedforward Networks in Transformer Models
 2. raw_bm25=11.044  Pay Attention to What You Need
 3. raw_bm25=9.116  SpectFormer: Frequency and Attention is what you need in a Vision Transformer
 4. raw_bm25=7.740  An Attention Free Transformer
 5. raw_bm25=7.386  Graph-Aware Transformer: Is Attention All Graphs Need?
 6. raw_bm25=6.661  RecurFormer: Not All Transformer Heads Need Self-Attention
 7. raw_bm25=6.577  Multi-Cast Attention Networks for Retrieval-based Question Answering and Response Prediction
 8. raw_bm25=5.820  Conversational Question Answering over Knowledge Graphs with Transformer and Graph Attention Networks
 9. raw_bm25=5.723  Energy-Gated Attention and Wavelet Positional Encoding: Complementary Inductive Biases for Transformer Attention
10. raw_bm25=5.648  Selective Attention Improves Transformer


## 9. Hybrid search (semantic + keyword, normalized and combined)

In [14]:
hybrid_results = run_search(
    hybrid_searcher, SEARCH_QUERY, mode="hybrid", alpha=0.6, candidate_k=50, top_k=10
)
for r in hybrid_results:
    print(f"{r.rank:>2}. hybrid={r.hybrid_score:.3f}  {r.paper.title}")

 1. hybrid=0.639  Area Attention
 2. hybrid=0.573  Pay Attention to What You Need
 3. hybrid=0.400  Attention Is Not All You Need: The Importance of Feedforward Networks in Transformer Models
 4. hybrid=0.396  Selective Attention Improves Transformer
 5. hybrid=0.344  Top-Theta Attention: Sparsifying Transformers by Compensated Thresholding
 6. hybrid=0.317  Contrastive Attention Mechanism for Abstractive Sentence Summarization
 7. hybrid=0.313  Attention Flows: Analyzing and Comparing Attention Mechanisms in Language Models
 8. hybrid=0.285  Skip-Layer Attention: Bridging Abstract and Detailed Dependencies in Transformers
 9. hybrid=0.272  Do New Attention Mechanisms Actually Fix Attention Sinks at Million-Token Context?
10. hybrid=0.267  Analyzing the Structure of Attention in a Transformer Language Model


## 10. Hybrid + cross-encoder reranking, with meaningful 0-1 scores

`with_display_scores` converts raw scores (unbounded BM25, raw cross-encoder logits) into a consistent 0-1 scale for display: 1.0 = strongest match in this result set, 0.0 = weakest. This only changes presentation, while internal ranking always uses the original raw scores.

In [ ]:
reranker = CrossEncoderReranker()

reranked_results = run_search(
    hybrid_searcher,
    SEARCH_QUERY,
    mode="hybrid",
    alpha=0.6,
    candidate_k=50,
    top_k=10,
    rerank=True,
    reranker=reranker,
)

display_rows = with_display_scores(reranked_results)
for row in display_rows:
    r = row["result"]
    print(f"{r.rank:>2}. score={row['display_score']:.3f}  {r.paper.title}")
    print(f"     breakdown: {row['display_breakdown']}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4602.00it/s]


 1. score=0.997  Attention Is Not All You Need: The Importance of Feedforward Networks in Transformer Models
     breakdown: {'semantic': 0.0, 'keyword': 1.0, 'rerank': 0.9968765889622027}
 2. score=0.845  Graph-Aware Transformer: Is Attention All Graphs Need?
     breakdown: {'semantic': 0.0, 'keyword': 0.32777628620162047, 'rerank': 0.8447211445501571}
 3. score=0.644  SpectFormer: Frequency and Attention is what you need in a Vision Transformer
     breakdown: {'semantic': 0.0, 'keyword': 0.5430284482419838, 'rerank': 0.6437216911182977}
 4. score=0.370  Attention is not not Explanation
     breakdown: {'semantic': 0.3176083671504649, 'keyword': 0.02301132341694709, 'rerank': 0.3701796298450643}
 5. score=0.235  Area Attention
     breakdown: {'semantic': 1.0, 'keyword': 0.0, 'rerank': 0.23515731356918265}
 6. score=0.228  Pay Attention to What You Need
     breakdown: {'semantic': 0.41873783903451584, 'keyword': 0.7827225943622174, 'rerank': 0.22754442518784843}
 7. score=0.227  Re

## 11. Metadata filtering

Filter the local library by category and/or publication date before
searching.

In [16]:
filtered_papers = repository.all(category=CATEGORY, from_date="2022-01-01")
print(f"{len(filtered_papers)} papers in {CATEGORY} published on/after 2022-01-01")

filtered_bm25 = BM25Index()
filtered_bm25.build(filtered_papers)
filtered_hybrid = HybridSearcher(store=repository, bm25_index=filtered_bm25, searcher=searcher)

filtered_results = run_search(filtered_hybrid, SEARCH_QUERY, mode="hybrid", top_k=5)
for r in filtered_results:
    print(f"{r.rank}. {r.paper.title} ({r.paper.published})")

127 papers in cs.CL published on/after 2022-01-01
1. Pay Attention to What You Need (2023-07-25)
2. Area Attention (2018-10-23)
3. Attention Is Not All You Need: The Importance of Feedforward Networks in Transformer Models (2025-05-10)
4. Selective Attention Improves Transformer (2024-10-03)
5. Top-Theta Attention: Sparsifying Transformers by Compensated Thresholding (2025-02-12)


## 12. Index persistence: delete and rebuild without re-embedding

Demonstrates that the FAISS index is a disposable cache and deleting it loses nothing, since embeddings are safely stored in SQLite.

In [17]:
import os

os.remove(DEFAULT_INDEX_PATH)
os.remove(DEFAULT_ID_MAP_PATH)
print("Deleted the FAISS index files")

rebuilt_index = service.rebuild_index()
rebuilt_index.save(DEFAULT_INDEX_PATH, DEFAULT_ID_MAP_PATH)
print(f"Rebuilt index with {len(rebuilt_index)} vectors -- no re-embedding needed")

Deleted the FAISS index files
Rebuilt index with 200 vectors -- no re-embedding needed


## 13. Visualize: does reranking actually change the ranking?

Scatter plot of first-stage retrieval score vs. cross-encoder rerank score for the same candidates. Points near the dashed line agree with retrieval; points far from it (brighter color = larger rank change) were substantially reordered by reranking.

In [18]:
reranking_plot_path = plot_reranking_impact(
    reranked_results, "../data/plots/reranking_impact.png"
)
print(f"Saved {reranking_plot_path}")

Saved ../data/plots/reranking_impact.png


## 14. Visualize: how much do the three search modes actually overlap?

Bar chart showing, for the same query, how many top-10 papers are unique to semantic, keyword, or hybrid search versus shared across modes. A concrete check that hybrid search isn't just duplicating one of the others.

In [19]:
overlap_plot_path = plot_mode_overlap(
    semantic_results, keyword_results, hybrid_results, "../data/plots/mode_overlap.png"
)
print(f"Saved {overlap_plot_path}")

Saved ../data/plots/mode_overlap.png


## 15. Interactive explore: auto-ingest + hybrid+rerank search

`run_explore` is the same interactive loop behind `uv run -m paper_explorer explore`. It prompts for a query, checks whether the local library already covers that topic well (via a quick semantic check), automatically ingests 100 papers from arXiv if not, then searches with hybrid + reranking and prints results on a meaningful 0-1 score scale.

**This cell is interactive** -- it will pause and wait for you to type a query in the input box that appears below the cell. Type a query (e.g. `attention is all you need`), press Enter to see results, then type `quit` to exit the loop and let the cell finish running.

In [22]:
from paper_explorer.explore import run_explore

run_explore(
    db_path=DEFAULT_DB_PATH,
    index_path=DEFAULT_INDEX_PATH,
    id_map_path=DEFAULT_ID_MAP_PATH,
    threshold=0.35,
    top_k=10,
    candidate_k=50,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7187.64it/s]
/Users/sujan/Desktop/paper_finder/paper_explorer/src/paper_explorer/embeddings/embedding_model.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self._model.get_sentence_embedding_dimension()


Loading reranker (first run downloads a small model)...

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4565.41it/s]


Paper Explorer -- interactive search

Type a query, or 'quit' to exit.

...

https://arxiv.org/abs/2408.00039v1

Blood clotting is an important physiological process to suppress bleeding upon injury, but when it occurs 
inadvertently, it can cause thrombosis, which can lead to life threatening conditions. Hence, understanding the...

Search query:

Search query:

Goodbye!